# Scenario 13: Customer OTel Pattern Validation

This notebook validates the **exact tracing pattern from the customer's `min_manual_otel_fix.py`** against our Galileo backend. The goal is to confirm that manually-created OTel spans — with no OpenInference, no Galileo span helpers, and no LLM auto-instrumentation — are correctly ingested and classified by Galileo.

We run two tests:

1. **Mock LLM test** (Step 3) — reproduces the customer's code exactly: hardcoded response, hardcoded token counts, no OpenAI call. Isolates the OTel pipeline from LLM availability.
2. **Real LLM test** (Step 4) — same span structure, but with a real `client.chat.completions.create(...)` call. Proves end-to-end.

## What the customer's script does

| Component | Customer's choice |
|---|---|
| Span processor | `BatchSpanProcessor` + `OTLPSpanExporter` |
| OTLP endpoint | `{api_url}/otel/traces` |
| Auth | `Galileo-API-Key`, `project`, `logstream` headers |
| HTTP instrumentation | `RequestsInstrumentor` |
| LLM spans | Manual (`tracer.start_as_current_span`) |
| OpenInference | **None** |
| Galileo SDK in span path | **None** |
| SSL | Custom CA cert via `SSL_CERT_FILE` + `REQUESTS_CA_BUNDLE` |

## How this relates to Scenarios 10–12

| Aspect | S10 | S11 | S12 | **S13 (this)** |
|---|---|---|---|---|
| LLM spans | Auto (OpenInference) | Auto (OpenInference) | Manual | **Manual + mock** |
| Exporter | GalileoSpanProcessor | Raw OTLP | Raw OTLP | **Raw OTLP** |
| Focus | Galileo helpers | Pure OTel | Full manual | **Customer validation** |

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY`
- Dependencies installed via `uv sync`

## Step 0: Load Environment Variables

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


## Step 0b: Imports and Constants

These imports **exactly match** the customer's `min_manual_otel_fix.py`. There is no OpenInference, no `galileo.otel`, and no `WorkflowSpan`/`ToolSpan`/`RetrieverSpan` schema classes. The only Galileo imports are for project setup and teardown.

### Customer's enterprise config (adapted)

The customer's script hardcodes `GALILEO_CONSOLE_URL`, `GALILEO_API_URL`, `SSL_CERT_FILE`, and `GALILEO_API_KEY`. We read these from `.env` and `GalileoPythonConfig` instead, but the OTel pipeline configuration is identical.

> **SSL note:** The customer sets `SSL_CERT_FILE` and `REQUESTS_CA_BUNDLE` for their enterprise CA cert. We skip this in the notebook since our demo environment uses public CAs. If testing against an on-prem Galileo instance, uncomment the SSL lines below.

In [2]:
import json
import os
from urllib.parse import urljoin

import openai
from galileo import galileo_context
from galileo.config import GalileoPythonConfig
from galileo.projects import delete_project
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.instrumentation.requests import RequestsInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

PROJECT_NAME = 'GalileoEval_S13_CustomerOTel'
LOG_STREAM_NAME = 'customer-otel-validation'
SERVICE_NAME = 'galileo-customer-otel-validation'
MODEL = 'gpt-4o-mini'

# -- Uncomment for enterprise/on-prem with custom CA cert --
# SSL_CERT_FILE = './your-enterprise-ca.pem'
# os.environ['SSL_CERT_FILE'] = SSL_CERT_FILE
# os.environ['REQUESTS_CA_BUNDLE'] = SSL_CERT_FILE

PROJECT_NAME, LOG_STREAM_NAME, SERVICE_NAME, MODEL

('GalileoEval_S13_CustomerOTel',
 'customer-otel-validation',
 'galileo-customer-otel-validation',
 'gpt-4o-mini')

## Step 1: Initialize Galileo (project + log stream only)

`galileo_context.init()` creates the project and log stream. We read back `api_url` and `api_key` from `GalileoPythonConfig` to wire into the raw OTLP exporter.

The customer's script does the same thing but reads these from hardcoded constants. The OTel pipeline that follows is identical either way.

In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

GALILEO_API_URL = str(config.api_url)
GALILEO_API_KEY = config.api_key.get_secret_value() if config.api_key else None

OTEL_TRACES_ENDPOINT = os.environ.get(
    'OTEL_EXPORTER_OTLP_TRACES_ENDPOINT',
    urljoin(GALILEO_API_URL if GALILEO_API_URL.endswith('/') else GALILEO_API_URL + '/', 'otel/traces'),
)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    log_stream_url = 'Galileo context initialized, but no log stream URL was returned.'
    print(log_stream_url)

print(f'\nOTLP endpoint -> {OTEL_TRACES_ENDPOINT}')

https://console.demo-v2.galileocloud.io/project/f7391db1-d112-4003-aa3e-1e04bc6b6ca7
https://console.demo-v2.galileocloud.io/project/f7391db1-d112-4003-aa3e-1e04bc6b6ca7/log-streams/5c17ab8d-f664-454e-a61d-b5d85ea720ea

OTLP endpoint -> https://api.demo-v2.galileocloud.io/otel/traces


## Step 2: Build the OTel Pipeline (Customer's `configure_tracing()`)

This is a **line-for-line match** of the customer's pipeline setup:

1. `Resource` with `service.name`
2. `TracerProvider`
3. `OTLPSpanExporter` with Galileo endpoint + auth headers
4. `BatchSpanProcessor`
5. `RequestsInstrumentor` for outbound HTTP spans

The tracer name `fin-workflow` matches the customer's code.

In [ ]:
def configure_tracing():
    resource = Resource.create({
        'service.name': SERVICE_NAME,
        'service.version': '1.0.0',
        'deployment.environment': 'notebook',
    })

    tracer_provider = TracerProvider(resource=resource)

    exporter = OTLPSpanExporter(
        endpoint=OTEL_TRACES_ENDPOINT,
        headers={
            'Galileo-API-Key': GALILEO_API_KEY,
            'project': PROJECT_NAME,
            'logstream': LOG_STREAM_NAME,
        },
    )

    span_processor = BatchSpanProcessor(exporter)
    tracer_provider.add_span_processor(span_processor)
    trace.set_tracer_provider(tracer_provider)

    req_instrumentor = RequestsInstrumentor()
    try:
        req_instrumentor.uninstrument()
    except Exception:
        pass
    req_instrumentor.instrument(tracer_provider=tracer_provider)

    return tracer_provider, span_processor, req_instrumentor


tracer_provider, span_processor, req_instrumentor = configure_tracing()
tracer = trace.get_tracer('fin-workflow')
client = openai.OpenAI()

print('OTel pipeline ready (customer pattern):')
print(f'  Service      -> {SERVICE_NAME}')
print(f'  Endpoint     -> {OTEL_TRACES_ENDPOINT}')
print(f'  Tracer       -> fin-workflow')
print('  Instrumentor -> RequestsInstrumentor (HTTP only, no LLM auto-instrumentation)')

## Step 3: Mock LLM Test (Customer's Exact Code)

This cell reproduces the customer's `min_manual_otel_fix.py` **as closely as possible**:

- The OpenAI call is replaced with `answer = 'this is the sample text'`
- Token counts are hardcoded to `10000` input / `5000` output
- Span names use the customer's `workflow:{name}`, `retriever:{name}`, `tool:{name}` convention
- Business attributes: `workflow.name`, `usecase.id`, `sys.id`, `step.name`, `step.operation`

If this trace appears correctly in the Galileo console with the right span types (workflow, retriever, LLM, tool), the customer's OTel pipeline is working.

In [ ]:
user_question = 'What are the best practices for OpenTelemetry in LLM apps?'
workflow_name = 'research-agent'

with tracer.start_as_current_span(
    f'workflow:{workflow_name}',
    attributes={
        'gen_ai.system': 'custom-otel',
        'workflow.name': workflow_name,
        'usecase.id': 'demo-usecase-001',
        'sys.id': 'notebook-sys-id',
        'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question}]),
    },
) as workflow_span:

    # 1) Retriever span
    retrieved_docs = [
        {'content': 'Always use a BatchSpanProcessor in production.', 'metadata': {'source': 'otel-docs', 'score': 0.91}},
        {'content': 'Set OTEL_SERVICE_NAME so spans carry a service identity.', 'metadata': {'source': 'otel-docs', 'score': 0.87}},
        {'content': 'Use vendor headers for OTLP routing instead of per-span attributes.', 'metadata': {'source': 'galileo-docs', 'score': 0.84}},
    ]
    with tracer.start_as_current_span(
        'retriever:vector-search',
        attributes={
            'gen_ai.system': 'custom-otel',
            'db.operation': 'search',
            'http.url': 'https://vector-store.internal/query',
            'http.method': 'POST',
            'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question}]),
        },
    ) as retriever_span:
        retriever_span.set_attribute(
            'gen_ai.output.messages',
            json.dumps([{'role': 'assistant', 'content': {'documents': retrieved_docs}}]),
        )
        print(f'Retrieved {len(retrieved_docs)} documents')

    # 2) Mock LLM span — NO OpenAI call, matches customer's code exactly
    context_text = '\n'.join(f'- {d["content"]}' for d in retrieved_docs)
    llm_messages = [
        {'role': 'system', 'content': 'Answer concisely based on the provided context.'},
        {'role': 'user', 'content': f'Context:\n{context_text}\n\nQuestion: {user_question}'},
    ]
    with tracer.start_as_current_span(
        'chat',
        attributes={
            'gen_ai.system': 'openai',
            'gen_ai.operation.name': 'chat',
            'gen_ai.request.model': MODEL,
            'gen_ai.input.messages': json.dumps(llm_messages),
        },
    ) as llm_span:
        # Customer's mock response — no actual LLM call
        answer = 'this is the sample text'
        llm_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'assistant', 'content': answer}]))
        llm_span.set_attribute('gen_ai.response.model', MODEL)
        llm_span.set_attribute('gen_ai.usage.input_tokens', 10000)
        llm_span.set_attribute('gen_ai.usage.output_tokens', 5000)

    # 3) Tool span
    step_name = 'format-final-answer'
    with tracer.start_as_current_span(
        f'tool:{step_name}',
        attributes={
            'gen_ai.system': 'custom-otel',
            'gen_ai.operation.name': 'execute_tool',
            'gen_ai.tool.name': step_name,
            'step.name': step_name,
            'step.operation': 'strip-whitespace',
            'gen_ai.tool.call.arguments': answer,
            'gen_ai.input.messages': json.dumps([{'role': 'tool', 'content': answer}]),
        },
    ) as tool_span:
        formatted = answer.strip()
        tool_span.set_attribute('gen_ai.tool.call.result', formatted)
        tool_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'tool', 'content': formatted}]))

    workflow_span.set_attribute(
        'gen_ai.output.messages',
        json.dumps([{'role': 'assistant', 'content': formatted}]),
    )

span_processor.force_flush()

print(f'\nMock answer: {formatted}')
print('\n-> MOCK trace: workflow -> retriever -> llm (mock) -> tool')
print(f'   Check: {log_stream_url}')
print('\nVerify in Galileo console:')
print('  - Workflow span named "workflow:research-agent"')
print('  - Retriever span named "retriever:vector-search" with 3 documents')
print('  - LLM span named "chat" with model gpt-4o-mini, 10000/5000 tokens')
print('  - Tool span named "tool:format-final-answer"')

## Step 4: Real LLM Test (End-to-End Validation)

Same span structure as Step 3, but now with a **real OpenAI call**. This proves the pattern works end-to-end: real LLM response, real token counts, real latency visible in Galileo.

If the customer's mock trace in Step 3 showed up correctly, this trace should too — the only difference is that the LLM span now has genuine content.

In [ ]:
user_question_real = 'Explain the difference between BatchSpanProcessor and SimpleSpanProcessor in OpenTelemetry.'
workflow_name_real = 'research-agent-real'

with tracer.start_as_current_span(
    f'workflow:{workflow_name_real}',
    attributes={
        'gen_ai.system': 'custom-otel',
        'workflow.name': workflow_name_real,
        'usecase.id': 'demo-usecase-002',
        'sys.id': 'notebook-sys-id',
        'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question_real}]),
    },
) as workflow_span:

    # 1) Retriever span
    retrieved_docs_real = [
        {'content': 'BatchSpanProcessor batches spans and exports them asynchronously for better performance.', 'metadata': {'source': 'otel-docs', 'score': 0.93}},
        {'content': 'SimpleSpanProcessor exports each span synchronously, useful for debugging.', 'metadata': {'source': 'otel-docs', 'score': 0.89}},
        {'content': 'In production, always prefer BatchSpanProcessor to avoid blocking the application.', 'metadata': {'source': 'otel-best-practices', 'score': 0.85}},
    ]
    with tracer.start_as_current_span(
        'retriever:vector-search',
        attributes={
            'gen_ai.system': 'custom-otel',
            'db.operation': 'search',
            'http.url': 'https://vector-store.internal/query',
            'http.method': 'POST',
            'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question_real}]),
        },
    ) as retriever_span:
        retriever_span.set_attribute(
            'gen_ai.output.messages',
            json.dumps([{'role': 'assistant', 'content': {'documents': retrieved_docs_real}}]),
        )
        print(f'Retrieved {len(retrieved_docs_real)} documents')

    # 2) REAL LLM span — actual OpenAI call
    context_text_real = '\n'.join(f'- {d["content"]}' for d in retrieved_docs_real)
    llm_messages_real = [
        {'role': 'system', 'content': 'Answer concisely based on the provided context.'},
        {'role': 'user', 'content': f'Context:\n{context_text_real}\n\nQuestion: {user_question_real}'},
    ]
    with tracer.start_as_current_span(
        'chat',
        attributes={
            'gen_ai.system': 'openai',
            'gen_ai.operation.name': 'chat',
            'gen_ai.request.model': MODEL,
            'gen_ai.input.messages': json.dumps(llm_messages_real),
        },
    ) as llm_span:
        synthesis = client.chat.completions.create(
            model=MODEL,
            messages=llm_messages_real,
            max_tokens=150,
        )
        answer_real = synthesis.choices[0].message.content
        llm_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'assistant', 'content': answer_real}]))
        llm_span.set_attribute('gen_ai.response.model', synthesis.model)
        llm_span.set_attribute('gen_ai.usage.input_tokens', synthesis.usage.prompt_tokens)
        llm_span.set_attribute('gen_ai.usage.output_tokens', synthesis.usage.completion_tokens)

    # 3) Tool span
    step_name_real = 'format-final-answer'
    with tracer.start_as_current_span(
        f'tool:{step_name_real}',
        attributes={
            'gen_ai.system': 'custom-otel',
            'gen_ai.operation.name': 'execute_tool',
            'gen_ai.tool.name': step_name_real,
            'step.name': step_name_real,
            'step.operation': 'strip-whitespace',
            'gen_ai.tool.call.arguments': answer_real,
            'gen_ai.input.messages': json.dumps([{'role': 'tool', 'content': answer_real}]),
        },
    ) as tool_span:
        formatted_real = answer_real.strip()
        tool_span.set_attribute('gen_ai.tool.call.result', formatted_real)
        tool_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'tool', 'content': formatted_real}]))

    workflow_span.set_attribute(
        'gen_ai.output.messages',
        json.dumps([{'role': 'assistant', 'content': formatted_real}]),
    )

span_processor.force_flush()

print(f'\nReal answer: {formatted_real}')
print(f'\nTokens: {synthesis.usage.prompt_tokens} in / {synthesis.usage.completion_tokens} out')
print('\n-> REAL trace: workflow -> retriever -> llm (real OpenAI) -> tool')
print(f'   Check: {log_stream_url}')

## Step 5: Standalone LLM Span (Simplest Possible Test)

If the nested traces above don't appear, this cell isolates the problem: one LLM span, no nesting. If **this** shows up in Galileo but the nested traces don't, the issue is in span nesting or attribute formatting, not the OTel pipeline itself.

In [ ]:
standalone_messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': 'Say hello in three languages.'},
]

with tracer.start_as_current_span(
    'chat',
    attributes={
        'gen_ai.system': 'openai',
        'gen_ai.operation.name': 'chat',
        'gen_ai.request.model': MODEL,
        'gen_ai.input.messages': json.dumps(standalone_messages),
    },
) as llm_span:
    standalone_resp = client.chat.completions.create(
        model=MODEL,
        messages=standalone_messages,
        max_tokens=60,
    )
    standalone_answer = standalone_resp.choices[0].message.content
    llm_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'assistant', 'content': standalone_answer}]))
    llm_span.set_attribute('gen_ai.response.model', standalone_resp.model)
    llm_span.set_attribute('gen_ai.usage.input_tokens', standalone_resp.usage.prompt_tokens)
    llm_span.set_attribute('gen_ai.usage.output_tokens', standalone_resp.usage.completion_tokens)

span_processor.force_flush()

print(f'Standalone: {standalone_answer}')
print(f'Tokens: {standalone_resp.usage.prompt_tokens} in / {standalone_resp.usage.completion_tokens} out')
print(f'\n-> Standalone LLM span flushed. Check: {log_stream_url}')

## Step 6: Flush and Verify

Final flush to make sure all traces are visible. After this cell, open the Galileo console link and verify:

1. **3 traces** visible in the log stream
2. **Trace 1 (mock):** workflow → retriever → LLM (mock, 10000/5000 tokens) → tool
3. **Trace 2 (real):** workflow → retriever → LLM (real OpenAI response) → tool
4. **Trace 3 (standalone):** single LLM span
5. Span types correctly classified (workflow, retriever, llm, tool)

In [ ]:
span_processor.force_flush()

print('All spans flushed to Galileo')
print(f'View traces at: {log_stream_url}')
print()
print('Expected traces:')
print('  1. workflow:research-agent         -> mock LLM (10000/5000 tokens)')
print('  2. workflow:research-agent-real     -> real OpenAI call')
print('  3. standalone chat                  -> single LLM span')

## Step 7: Clean Up

1. Uninstrument `requests` so later notebook cells or re-runs start cleanly
2. Shut down the `TracerProvider` to drain the batch processor
3. Delete the Galileo project

> **Tip:** Comment out the `delete_project` line if you want to keep the traces in Galileo for manual inspection.

In [ ]:
req_instrumentor.uninstrument()
tracer_provider.shutdown()

# delete_project(name=PROJECT_NAME)
# print(f'Deleted project: {PROJECT_NAME}')
print(f'Project kept for inspection: {PROJECT_NAME}')
print(f'View at: {log_stream_url}')